# 🫡 모델 저장

이 모델을 다양한 위치(다른 노트북이나 모델 서버 포함)에서 사용할 수 있도록 저장해야 하므로, S3 호환 저장소에 업로드합니다.

>주의: 클러스터별 변수를 변경하지 않고 모든 셀을 한 번에 실행하지 마세요.

### 필요한 패키지 설치 및 업로드 함수 정의

`pip`에서 오류가 발생하더라도 걱정하지 않아도 됩니다. 어쨌든 모든 것이 잘 실행될 것입니다.

In [ ]:
!pip -q install model-registry==0.2.15

# 🤩 쿠브플로우 레지스트리

우리는 구축하는 모델의 버전, 작성자, 모델 위치 등의 정보를 저장하기 위한 메타데이터 레지스트리가 필요합니다.

우리는 쿠브플로우 모델 레지스트리를 정규 데이터 소스로 사용하여 이러한 정보를 저장합니다.

레지스트리를 사용하는 이유는 다음과 같습니다 (_쿠브플로우 웹사이트에서_):

- 저장소에서 사용 가능한 모델 추적: 모델이 저장되면 쿠브플로우 모델 레지스트리에서 추적하여 라이프사이클을 관리할 수 있습니다. 모델 레지스트리는 이 정보를 카탈로그화, 나열, 인덱싱, 공유, 기록, 정렬할 수 있습니다. 이를 통해 데이터 과학자는 다양한 버전을 비교하고 필요시 이전 버전으로 되돌릴 수 있습니다.

- 성능 추적 및 비교: 각 모델 버전의 정확도, 재현율, 정밀도 같은 주요 메트릭을 봅니다. 이는 배포를 위한 최고 성능 모델을 식별하는 데 도움이 됩니다.

- 선형도(lineage) 만들기: 데이터, 코드, 모델 간의 관계를 캡처합니다. 이를 통해 데이터 과학자는 각 모델의 출처를 이해하고 특정 실험을 재현할 수 있습니다.

- 협업: 모델과 실험 세부 사항을 MLOps 엔지니어와 공유하여 배포 준비를 합니다. 이는 학습에서 프로덕션으로의 매끄러운 전환을 보장합니다.

레지스트리의 인스턴스는 개발 환경에서도 사용 가능합니다.

# 🪣 S3 저장소

모델 레지스트리의 백엔드로 S3 저장소를 사용합니다. 이는 우리의 모델이 S3에 저장되고 모델 레지스트리가 그들의 위치를 추적한다는 의미입니다.

쿠브플로우 모델 레지스트리 덕분에 우리의 모델을 S3 저장소와 모델 레지스트리에 동시에 푸시할 수 있으므로, 새로운 모델을 쉽게 저장하고 추적할 수 있습니다 🕵️

In [ ]:
from model_registry import ModelRegistry
from model_registry.utils import S3Params
from model_registry.exceptions import StoreError
import os

‼️⚠️ 중요 ⚠️‼️

이전에 공유받은 사용자 이름과 클러스터 도메인(apps.xxx)을 추가하세요. 모델 레지스트리 URL에 필요합니다.

In [ ]:
# 이전에 공유받은 사용자 이름과 클러스터 도메인(apps.xxx)을 추가하세요

username = "<USER_NAME>"
cluster_domain = "<CLUSTER_DOMAIN>"

In [ ]:
# 모델 레지스트리 연결 설정
model_registry_url = f"https://{username}-registry-rest.{cluster_domain}"
author_name = username

registry = ModelRegistry(server_address=model_registry_url, port=443, author=author_name, is_secure=False)

In [ ]:
# 등록하려는 모델 세부 정보
registered_model_name = "jukebox"
version = "0.0.1"
model_path = "models/jukebox/"

#### 🙏 데이터 연결 덕분에 S3 버킷 자격증명이 노트북에서 사용 가능합니다!

S3 버킷을 명시적으로 가져오지만, 다른 것들은 우리가 모델을 업로드하고 등록할 때 백그라운드에서 자동으로 사용됩니다 🧙‍♂️

In [ ]:
s3_upload_params = S3Params(
    bucket_name=os.environ.get('AWS_S3_BUCKET'),
    s3_prefix="models/jukebox/",
)

try:
    registered_model = registry.upload_artifact_and_register_model(
        name=registered_model_name,
        model_files_path=model_path,
        model_format_name="onnx",
        author=username,
        model_format_version="1",
        version=version,
        description="음악 데이터로 학습된 밀집 신경망",
        metadata={
            "accuracy": 0.3,
            "license": "apache-2.0"
        },
        upload_params=s3_upload_params
    )

    print(f"'{registered_model_name}' 버전 '{version}'이(가) 여기에 등록되었습니다: https://rhods-dashboard-redhat-ods-applications.{cluster_domain}/modelRegistry/{username}-registry/registeredModels/1/versions/{registry.get_model_version(registered_model_name, version).id}/details")

except StoreError:
    rmver = registry.get_model_version(registered_model_name, version)
    print(f"모델과 버전이 이미 존재합니다:\n{rmver}")

### 퀴즈 시간 🤓

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../.dontlookhere/'))
try: from quiz2 import *
except: pass

In [ ]:
try: quiz_versioning()
except: pass

### 모델 레지스트리에서 모델에 대한 좋은 정보도 얻을 수 있습니다

In [ ]:
# 등록된 모델의 일반 정보 출력
model = registry.get_registered_model("jukebox")
print("등록된 모델:", model, "ID:", model.id)

In [ ]:
# 등록된 모델의 버전 정보 출력
version = registry.get_model_version("jukebox", "0.0.1")
print("모델 버전:", version, "ID:", version.id)

In [ ]:
# 등록된 모델의 아티팩트 정보 출력
art = registry.get_model_artifact("jukebox", "0.0.1")
print("모델 아티팩트:", art, "ID:", art.id)

### 🥁 다음 단계

모델을 S3 저장소 및 레지스트리에 저장했으므로, 데이터 연결을 사용하여 모델을 참조하고 모델을 API로 제공할 수 있습니다.

지침으로 돌아가서 https://rhoai-mlops.github.io/lab-instructions/#/1-when-the-music-starts/4-inner-data-science-loop?id=model-serving 모델 레지스트리 UI에서 모델을 먼저 확인하세요.